# 04 · 호출한 건 하나인데 답은 넷으로 갈렸다 — 이 필드를 계속 물어볼까

## 지금 어디인가 — 전체에서 이 조각까지

```
PR 리뷰 멀티에이전트  ①웹훅 → ⑥게이트
 └─ M6 — ④⑤ 를 더미에서 진짜 LLM 으로
     └─ M6-4 배선 — 노드 하나를 넷으로 가른다 (security/quality/testing/docs)
         └─ 넷이 되면 "누가 찾았나"를 누가 정하나           ← 결정 D3
             └─ 그 값을 모델에게 계속 물어볼 것인가          ← 이 노트북
```

**윗층과 아랫층은 다른 근육이다.** 윗층(배선)은 "어떻게 연결하나"이고 손이 기억한다.
아랫층(이 노트북)은 **"이 필드가 있어서 얻는 게 뭔가"**이고, 그건 세어봐야 안다.

형제 노트북: `01` 은 비율을 재는 자(신뢰구간)를 만들었고, `03` 은 채점기가 실제로
어떻게 도는지 밟았다. **이 노트북은 01 에서 만든 자를 다시 꺼내 쓴다** — §3 에서.

## 왜 이걸 하나

**① 지금 우리 코드에서 벌어진 일**

`evals/runs/` 에 18판이 쌓여 있다. 그중 6판이 finding 마다 `agent_type` 을 같이 적었고,
그 17개가 이렇게 갈렸다:

```
security 7  ·  quality 6  ·  docs 2  ·  testing 2
```

넷으로 골고루 갈렸다. **그런데 그 6판을 만든 에이전트는 하나다.** 우리 코드에는 지금
`review_diff()` 하나뿐이고, 그게 "넷을 모두 훑어라"라는 프롬프트 하나로 돌았다.
호출한 쪽이 넷으로 갈린 적은 **한 번도 없다.**

**② 왜 곤란한가**

M6-4 가 노드를 진짜로 넷으로 가르면 이런 일이 생길 수 있다:

```
security 노드를 호출  →  모델이 agent_type="docs" 인 finding 을 반환
```

그럼 결과만 보면 **security 가 0개**다. 그런데 원인이 둘이다 —
`security 노드가 죽어서 0개`인가, `찾을 게 원래 없어서 0개`인가.
M8 게이트가 사람에게 넘길지 자동 게시할지 정할 때 이걸 물어야 하는데(**G2 커버리지 판정**),
출처가 틀리면 **게이트가 조용히 거짓말한다.**

D3 는 이미 *"`agent_type` 은 출처이고 코드가 정한다"*로 결정됐다. 프롬프트에서는 뺐다.
**그런데 스키마는 아직 그대로**라 모델이 여전히 값을 뱉는다. 지금은
*"묻지는 않지만 답은 계속 요구하는"* 어중간한 상태다.

**③ 그래서 뭘 하고 싶은가**

> 모델이 정할 수 없는 값을 **계속 요구할지** 정한다. 그리고 그 결정의 대가를 숫자로 만든다.

**④ 도구**

- **몬테카를로 시뮬레이션** — 참값(어느 노드가 진짜 죽었나)을 우리가 정해놓고,
  게이트가 그걸 맞히는지 수천 번 세어본다
- **비율의 신뢰구간** — 노트북 01 에서 만든 `wilson_ci` 를 그대로 꺼내 쓴다

## 이 노트북이 끝나면

| 할 수 있게 되는 것 |
|---|
| "어긋남을 계측기로 쓰자"는 논거가 **언제 성립하고 언제 안 하는지** 구분한다 |
| 게이트의 커버리지 판정이 **어떤 조건에서 거짓말하는지** 직접 세어서 보인다 |
| 세 갈래(그냥 두기 / 덮어쓰기 / 스키마에서 빼기)의 대가를 각각 한 줄로 말한다 |

| 안 다루는 것 |
|---|
| 프롬프트를 어떻게 고칠지 (그건 M6-3b) |
| LangGraph 노드를 실제로 배선하는 법 (그건 M6-4 코드) |
| 애그리게이터의 중복 제거 규칙 (그건 M6-5) |

**분량**: TODO 3개 · 20~30분
**의존성 정책**: `numpy` + `matplotlib` 만. `scipy` 는 **일부러 안 깐다** —
`scipy.stats` 가 있으면 §3 이 한 줄로 끝나고, 그 한 줄이 정확히 없애야 할 추상화다.

## 기호와 말

| 기호 / 말 | 읽는 법 | 뜻 한 줄 |
|---|---|---|
| **출처** | — | "이 지적을 **누가** 찾았나". 코드는 어느 노드를 불렀는지 안다 |
| **분류** | — | "이 지적이 **무슨 종류**인가". `category` 필드가 답한다 |
| **G2** | "지 투" | M8 게이트의 커버리지 판정 — *"이 관점이 0개인데, 죽어서인가 원래 없어서인가"* |
| `m` | "엠" | **어긋남률** — 모델이 출처를 틀리게 적을 확률 (0~1) |
| `m̂` | "엠 햇" | 실제로 세어본 어긋남률. 참값 `m` 과 다르다 — 표본이라서 |
| `K` | "케이" | 판 수. 같은 diff 를 몇 번 돌렸나 |
| **b0 / b1 / b2** | — | 세 갈래: 그냥 둠 / 모델값을 코드가 덮어씀 / 스키마에서 뺌 |

---
## 준비 — 레포 루트를 찾고 폰트를 잡는다

In [1]:
import sys, json
from pathlib import Path
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager, rcParams

ROOT = Path.cwd()
while not (ROOT / "fixtures").exists():      # 노트북을 어디서 열든 루트를 찾는다
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

# 한글 폰트 (없으면 그냥 넘어간다 — 그래프 라벨만 깨진다)
_avail = {f.name for f in font_manager.fontManager.ttflist}
for _f in ["AppleGothic", "Apple SD Gothic Neo", "NanumGothic", "Malgun Gothic"]:
    if _f in _avail:
        rcParams["font.family"] = _f
        break
rcParams["axes.unicode_minus"] = False
rcParams["figure.dpi"] = 110

rng = np.random.default_rng(20260828)
AGENTS = ["security", "quality", "testing", "docs"]

print("레포 루트:", ROOT)

레포 루트: /Users/imseungmin/work/llm_study/pr_agent_project


---
# §0 · 그 숫자를 직접 꺼낸다

브리핑의 `security 7 · quality 6 · docs 2 · testing 2` 를 **손으로 옮겨 적지 않는다.**
파일에서 읽는다 — 옮겨 적는 순간 오타가 사실이 된다.

In [2]:
runs_dir = ROOT / "evals" / "runs"

all_findings = []        # (파일명, 판번호, finding dict)
for path in sorted(runs_dir.glob("*.json")):
    data = json.loads(path.read_text())
    for i, run in enumerate(data["runs"]):
        for fd in run.get("findings") or []:
            all_findings.append((path.name, i, fd))

typed = [(f, i, fd) for f, i, fd in all_findings if "agent_type" in fd]

print(f"finding 총 {len(all_findings)}개")
print(f"그중 agent_type 이 기록된 것 {len(typed)}개 "
      f"({len(typed)/len(all_findings):.0%})")
print()
print("분포:", dict(Counter(fd["agent_type"] for _, _, fd in typed)))
print()
print("어느 판에서 나왔나:")
for key, n in sorted(Counter((f, i) for f, i, _ in typed).items()):
    print(f"  {key[0]}  run{key[1]}")

finding 총 53개
그중 agent_type 이 기록된 것 17개 (32%)

분포: {'security': 7, 'quality': 6, 'docs': 2, 'testing': 2}

어느 판에서 나왔나:
  sample__luna__orig__k9.json  run3
  sample__luna__orig__k9.json  run4
  sample__luna__orig__k9.json  run5
  sample__luna__orig__k9.json  run6
  sample__luna__orig__k9.json  run7
  sample__luna__orig__k9.json  run8


### 🖐 멈추고 볼 것

위 출력에서 **판(run)이 몇 개인지** 세어라. 그리고 이 사실과 붙여라:

> 그 판들은 전부 `backend/agents/base.py` 의 `review_diff()` **하나**가 냈다.
> 그 함수는 인자가 `diff_text` 하나뿐이다 — **어느 에이전트인지 넘길 자리가 없다.**

`security 7 · quality 6` 은 **넷이 각자 일한 결과가 아니다.**
하나가 넷인 척한 결과다.

---
# §1 · "어긋남을 계측기로 쓰자" — 이 논거를 시험한다

세 갈래 중 **b1(덮어쓰기)** 의 논거는 이것이다:

> 스키마에 필드를 남겨두면 모델이 계속 값을 뱉는다.
> 코드가 그 위에 진짜 출처를 덮어쓰되, **덮어쓰기 전에 두 값을 비교해 로그로 남기면**
> "프롬프트가 얼마나 새는가"를 계속 잴 수 있다. **공짜 계측기다.**

그럴듯하다. 그러면 **지금 데이터로 그 계측기를 한번 돌려보자.**

### 🔨 TODO(human) ① — 어긋남을 무엇으로 세나

`count_mismatch(findings, called_agent)` 를 채운다.
`findings` 는 finding dict 의 리스트, `called_agent` 는 **코드가 실제로 부른 노드 이름**.
`(어긋난 개수, 전체 개수)` 를 돌려준다.

**후보 셋 — 무엇을 "어긋남"으로 볼 것인가:**

| | 세는 것 | 이 축을 고르면 |
|---|---|---|
| (a) | `fd["agent_type"] != called_agent` | 출처가 틀린 것만 센다 |
| (b) | `fd["agent_type"]` 이 `fd["category"]` 와 안 어울림 | 분류가 틀린 것을 센다 |
| (c) | (a) 또는 (b) | 둘 다 |

**고르는 기준**: D3 가 이 필드를 무엇으로 확정했나 — **출처**인가 **분류**인가?
그 답이 후보를 하나로 좁힌다. (b) 를 고르려면 "어울림"을 누가 판정하는지 먼저 답해야 한다.

**힌트**: `agent_type` 키가 없는 finding 도 섞여 있다. 그건 세지 않는다 —
전체 개수에도 넣지 않는다. "기록이 없다"와 "안 어긋났다"는 다르다.

**채우기 전에 돌리면**: `TypeError: cannot unpack non-sequence NoneType` 이 난다.
오류가 아니라 **빈칸이 비었다는 신호**다.

**검산**: 같은 `findings` 에 `called_agent` 를 바꿔 넣으면 **결과도 바뀌어야 한다.**
안 바뀌면 `called_agent` 를 안 쓰고 있는 것이다.

In [ ]:
def count_mismatch(findings, called_agent):
    """출처가 어긋난 finding 을 센다. (어긋난 개수, 전체 개수) 를 돌려준다.

    findings    : finding dict 의 리스트. agent_type 키가 없는 것도 섞여 있다
    called_agent: 코드가 실제로 부른 노드 이름 ("security" 등)
    """
    ...


# 위 §0 에서 모은 17개로 돌려본다.
only_typed = [fd for _, _, fd in typed]

for called in AGENTS:
    bad, total = count_mismatch(only_typed, called)
    print(f"called={called:9s} → 어긋남 {bad}/{total} = {bad/total:.0%}")

TypeError: cannot unpack non-iterable NoneType object

### 🖐 예측 → 확인

위 셀을 돌리기 **전에** 적어라:

> `called="security"` 로 재면 어긋남률이 얼마쯤 나올까?
> (a) 0% 근처 (b) 50% 근처 (c) 60% 이상 · **왜 그렇게 생각했나 한 줄:**

그리고 돌린 뒤 **진짜 질문**은 이것이다:

> 네 줄의 숫자가 전부 다르다. **그중 어느 줄이 맞는 줄인가?**

### 💡 §1 의 반례 한 장면

답: **맞는 줄이 없다.**

`called_agent` 는 "코드가 부른 노드"인데, 이 17개를 만든 코드는 **노드를 부른 적이 없다.**
`review_diff(diff_text)` 하나가 "넷을 모두 훑어라"로 돌았을 뿐이다.
그러니 네 줄은 전부 **없는 질문에 대한 답**이다.

**그래서 b1 의 논거가 무너지는 자리가 여기다:**

> "계측기를 남겨두자"는 **계측 대상이 존재할 때만** 성립한다.
> 어긋남은 *호출자가 넷으로 갈린 뒤에야* 정의된다.
> 즉 b1 을 고르는 근거는 **아직 존재하지 않는 신호**다.

⚠️ **b1 이 틀렸다는 뜻은 아니다.** 배선이 끝나면 그 신호는 진짜로 생긴다.
지금 확인한 건 *"지금 있는 데이터가 b1 을 지지하지 않는다"* 이다. 둘은 다른 말이다.
§3 에서 **배선 후에 그 신호가 쓸 만할지**를 미리 재본다.

---
# §2 · 게이트는 언제 거짓말하나

M8 게이트의 **G2 커버리지 판정**이 묻는 것:

> `security` 관점의 finding 이 0개다. **security 노드가 죽어서인가, 찾을 게 없어서인가?**

앞은 "사람이 봐야 한다"이고 뒤는 "통과시켜도 된다"다. **정반대의 결론**이다.

판정 재료는 둘:
- `failed_agents` — 오케스트레이터가 아는 것. 예외가 났거나 타임아웃 난 노드 이름
- finding 들의 **출처** — 이게 지금 문제의 그 필드다

### 🔨 TODO(human) ② — G2 판정 함수

`g2_verdict(findings, failed, key)` 를 채운다.
`AGENTS` 넷 각각에 대해 셋 중 하나를 담은 dict 를 돌려준다:

| 값 | 뜻 |
|---|---|
| `"죽음"` | 노드가 실패했다 → 사람에게 넘긴다 |
| `"없음"` | 노드는 살았는데 찾은 게 0개다 → 통과 가능 |
| `"찾음"` | 노드가 finding 을 냈다 |

인자:
- `findings` : dict 리스트. **`"by_code"` 와 `"by_model"` 두 키를 다 갖고 있다**
- `failed`   : 실패한 노드 이름의 집합 (set)
- `key`      : 출처를 어느 값으로 볼지 — `"by_code"` 또는 `"by_model"`

**판단이 갈리는 자리**: 세 값의 **우선순위**다.
어떤 노드가 `failed` 에 있는데 그 이름의 finding 도 있으면 뭐라고 답할 것인가?

| | 우선순위 | 이러면 |
|---|---|---|
| (a) | `failed` 를 먼저 본다 | 실패한 노드는 finding 이 있어도 `"죽음"` |
| (b) | finding 을 먼저 본다 | 뭐라도 냈으면 `"찾음"` |

**고르는 기준**: 이 프로젝트의 **제1원칙이 선별(틀린 말을 안 하는 것)** 이라는 것.
둘 중 어느 쪽이 "모르는데 안다고 말하는" 실수를 덜 하나?

**틀리면**: (b) 를 고르면 타임아웃 직전에 하나 뱉고 죽은 노드가 `"찾음"` 이 되어
**부분 결과를 완전한 결과로 읽는다.**

**검산**: `failed` 를 비우고 `findings` 도 비우면 넷 다 `"없음"` 이어야 한다.

In [ ]:
def g2_verdict(findings, failed, key):
    """AGENTS 넷 각각에 대해 "죽음" / "없음" / "찾음" 을 판정한다.

    findings: dict 리스트. 각각 "by_code" 와 "by_model" 키를 갖는다
    failed  : 실패한 노드 이름의 set
    key     : 출처를 어느 값으로 볼지 — "by_code" 또는 "by_model"
    """
    ...


# 손으로 만든 장면 하나 — security 노드가 낸 것을 모델이 docs 라고 적었다.
scene = [
    {"by_code": "security", "by_model": "docs",    "category": "sql-injection"},
    {"by_code": "quality",  "by_model": "quality", "category": "resource-leak"},
]
print("코드가 붙인 출처로 :", g2_verdict(scene, failed=set(), key="by_code"))
print("모델이 적은 출처로 :", g2_verdict(scene, failed=set(), key="by_model"))

# 검산 — 아무것도 없으면 넷 다 "없음"
assert set(g2_verdict([], failed=set(), key="by_code").values()) == {"없음"}
print("\n검산 통과")

### 🖐 두 줄을 나란히 놓고 볼 것

`security` 칸이 두 줄에서 다르게 나온다.

- 코드가 붙인 출처: `security` → `"찾음"` ✅ 사실이다
- 모델이 적은 출처: `security` → `"없음"` ❌ **SQL 인젝션을 찾아놓고 "못 찾았다"**

그리고 `docs` 칸은 반대로 뒤집힌다. **한 번의 어긋남이 두 칸을 동시에 망친다.**

이게 게이트가 거짓말하는 방식이다. 에러도 경고도 안 난다 — **조용히 틀린다.**

---
# §3 · 그 계측기, 배선 후엔 쓸 만한가

§1 에서 b1 의 논거가 *"지금은"* 성립 안 한다고 했다. 그럼 배선이 끝난 뒤엔?

배선 후에는 어긋남 `m` 을 진짜로 잴 수 있다. 그런데 **몇 판을 돌려야 그 숫자가
말을 하나?** 여기서 노트북 01 의 자를 다시 꺼낸다 — `wilson_ci`.

⚠️ 새로 구현하지 않는다. **이미 만들어서 `evals/stats.py` 에 옮겨둔 것**을 import 한다.

In [ ]:
from evals.stats import wilson_ci

print("어긋남을 0건 관측했을 때, '어긋남률은 0%다' 라고 말할 수 있나?\n")
print(f"{'판당 finding':>12} {'n':>5} {'관측':>6}   Wilson 95% 구간        폭")
for n in [17, 40, 100, 300, 1000]:
    lo, hi = wilson_ci(0, n)
    print(f"{'':>12} {n:5d} {'0/'+str(n):>6}   [{lo:.3f}, {hi:.3f}]   {hi-lo:.3f}")

### 🖐 읽는 법

`n=17` 은 §0 에서 실제로 가진 데이터 양이다. 거기서 어긋남이 **0건** 나와도
구간 위쪽은 0 근처로 안 내려온다 — **"어긋남률이 20% 일 수도 있다"** 를 못 배제한다.

노트북 01 의 그 교훈이 여기서 돌아온다:

> 관측이 0/n 이라고 참값이 0인 게 아니다. **n 이 작으면 자가 아무 말도 못 한다.**

**b1 의 계측기가 값어치를 가지려면 n 이 수백은 돼야 한다.**
지금 K=3~9 로 돌리는 속도(판당 ~17초)로 그 n 에 언제 닿을지 계산해보면,
"공짜 계측기"라는 말이 얼마나 공짜인지 감이 온다.

---
# §4 · 참값을 정해놓고 세어본다

여기가 이 노트북의 핵심이다. **책도 나도 믿을 필요가 없다 — 세어보면 된다.**

⚠️ **먼저 범위를 좁힌다.** `"죽음"` 판정은 `failed_agents` 가 하는 일이다 —
오케스트레이터가 예외·타임아웃을 직접 보니까 `agent_type` 과 **아무 상관이 없다.**
어긋남이 실제로 망가뜨리는 건 **살아있는 노드들 사이의 `"찾음"` / `"없음"` 구분**이고,
그게 G2 가 묻는 바로 그것이다 — *"살아서 돌았는데 0개다. 통과시켜도 되나?"*

방법:
1. **우리가** 정한다 — 이번 판에서 어느 노드가 죽었나 (`dead`). 오케스트레이터는 이걸 안다
2. 살아있는 노드는 **확률 `p_find` 로** finding 을 하나 낸다 (찾을 게 없는 판도 있으니까)
3. 그 finding 의 `by_code` 는 항상 정확하다 (코드가 어느 노드를 불렀는지 아니까)
4. `by_model` 은 **확률 `m` 으로** 엉뚱한 노드 이름이 붙는다
5. **참값은 `by_code` 로 낸 판정 그 자체다.** 거기에 `by_model` 판정을 대보고 틀린 칸을 센다

그리고 틀린 칸을 **두 종류로 가른다** — 이게 이 시뮬레이션의 핵심이다:

| | 참값 | G2 가 말한 것 | 결과 |
|---|---|---|---|
| **놓침** | `"찾음"` | `"없음"` | 지적을 찾아놓고 **"깨끗하다"로 읽힌다** → 자동 게시된다 |
| **소란** | `"없음"` | `"찾음"` | 안 찾았는데 찾은 척 → 사람을 부른다 |

제1원칙이 **선별(틀린 말을 안 하는 것)** 이라 **놓침이 훨씬 비싸다.**

### 🔨 TODO(human) ③ — 한 판을 만든다

`one_trial(rng, m, n_dead, p_find)` 를 채운다. 돌려주는 것:

```
(dead, findings)
```

- `dead` : 이번 판에 죽은 노드 이름의 **set**. `AGENTS` 넷 중 `n_dead` 개를 고른다
- `findings` : dict 리스트 — `{"by_code": ..., "by_model": ...}`

**규칙**
- 죽은 노드는 아무것도 안 낸다
- 살아있는 노드는 **확률 `p_find` 로** finding 을 하나 낸다 (아니면 빈손)
- `by_code` 는 그 노드 이름 그대로
- `by_model` 은 확률 `m` 으로 **다른 셋 중 하나**, 아니면 노드 이름 그대로

**후보 — 어긋날 때 어디로 가나:**

| | 방식 | 이러면 |
|---|---|---|
| (a) | 나머지 셋 중 균등하게 | 가장 단순. 최악을 과소평가할 수도 |
| (b) | 항상 정해진 한 곳으로 (예: 늘 `security`) | 몰림. 실측 분포와 다르다 |
| (c) | §0 의 실측 분포를 따라 | 그럴듯하지만 그 분포는 단일 에이전트 것이다 |

**고르는 기준**: 이 시뮬레이션이 답하려는 질문이 *"m 이 커지면 G2 가 얼마나
망가지나"* 라는 것. **`m` 하나만 손잡이로 남기려면** 어느 방식이어야 하나?

**쓸 만한 것**: `rng.choice` · `rng.random()` · 파이썬 `set` 연산
**힌트**: "다른 셋 중 하나"를 고르려면 먼저 자기 자신을 후보에서 빼야 한다.

**검산 둘**:
- `m=0` 으로 두고 여러 판 돌리면 모든 finding 에서 `by_code == by_model`
- `p_find=1.0`, `n_dead=1` 이면 finding 이 **정확히 3개** (넷 중 하나가 죽었으니)

In [ ]:
def one_trial(rng, m, n_dead, p_find):
    """한 판을 만든다. (죽은 노드 set, finding 리스트) 를 돌려준다.

    rng   : numpy Generator
    m     : 어긋남률 — 모델이 출처를 틀리게 적을 확률 (0~1)
    n_dead: 이번 판에 죽는 노드 수 (0~4)
    p_find: 살아있는 노드가 finding 을 낼 확률 (0~1)
    """
    ...


# 검산 1 — m=0 이면 절대 어긋나지 않는다
_rng = np.random.default_rng(0)
for _ in range(200):
    _dead, _fs = one_trial(_rng, m=0.0, n_dead=1, p_find=0.7)
    assert len(_dead) == 1
    assert all(f["by_code"] not in _dead for f in _fs)
    assert all(f["by_code"] == f["by_model"] for f in _fs)

# 검산 2 — p_find=1.0 이면 살아있는 노드가 전부 낸다
_dead, _fs = one_trial(np.random.default_rng(3), m=0.0, n_dead=1, p_find=1.0)
assert len(_fs) == len(AGENTS) - 1, f"3개여야 하는데 {len(_fs)}개"
print("검산 통과 — m=0 에서 어긋남 0건, p_find=1 에서 살아있는 노드 수도 맞다")

# m 을 올리면 어긋나기 시작한다
_dead, _fs = one_trial(np.random.default_rng(1), m=0.9, n_dead=1, p_find=1.0)
print("\nm=0.9 인 한 판 · 죽은 노드:", _dead)
for f in _fs:
    mark = "  <- 어긋남" if f["by_code"] != f["by_model"] else ""
    print(f"  code={f['by_code']:9s} model={f['by_model']:9s}{mark}")

### 채점기 — 여기는 채워져 있다

**참값은 `by_code` 로 낸 판정 그 자체다.** 코드는 어느 노드를 불렀는지 아니까
그 판정은 정의상 옳다. 거기에 `by_model` 판정을 대보고 틀린 칸을 두 종류로 가른다.

In [ ]:
def score(dead, findings):
    '''(놓침 개수, 소란 개수) — 살아있는 노드 칸만 센다.'''
    truth = g2_verdict(findings, failed=dead, key="by_code")   # 정의상 옳다
    said  = g2_verdict(findings, failed=dead, key="by_model")
    missed = noisy = 0
    for a in AGENTS:
        if truth[a] == "찾음" and said[a] == "없음":
            missed += 1          # 찾아놓고 "깨끗하다" — 자동 게시된다
        elif truth[a] == "없음" and said[a] == "찾음":
            noisy += 1           # 안 찾았는데 찾은 척 — 사람을 부른다
    return missed, noisy


# ⚠️ failed=dead 를 양쪽에 똑같이 넘긴다. "죽음" 판정은 agent_type 과 무관하니까
#    두 판정에서 동일하게 상쇄되고, 남는 차이가 순수하게 출처 어긋남의 몫이 된다.
print("채점기 준비 완료")

In [ ]:
# ═══ 시뮬레이션 — m 을 훑으면서 두 종류의 오판을 센다 ═══
TRIALS = 4000
N_DEAD = 1
P_FIND = 0.7

ms = np.linspace(0, 0.5, 21)
missed_rate, noisy_rate = [], []

for m in ms:
    r = np.random.default_rng(42)
    tot_m = tot_n = 0
    for _ in range(TRIALS):
        dead, fs = one_trial(r, m, N_DEAD, P_FIND)
        a, b = score(dead, fs)
        tot_m += a
        tot_n += b
    missed_rate.append(tot_m / TRIALS)
    noisy_rate.append(tot_n / TRIALS)

fig, ax = plt.subplots(figsize=(7.2, 4.2))
ax.plot(ms, missed_rate, "o-", color="crimson", label="missed: found it, read as clean")
ax.plot(ms, noisy_rate, "s-", color="steelblue", label="noisy: found nothing, read as found")
ax.axhline(0, color="gray", lw=1, ls="--")
ax.set_xlabel("m  (probability the model mislabels the source)")
ax.set_ylabel("wrong verdicts per run")
ax.set_title(f"G2 coverage verdict when trusting the model's agent_type\n"
             f"(n_dead={N_DEAD}, p_find={P_FIND}, trials={TRIALS})")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

i10 = int(np.argmin(np.abs(ms - 0.10)))
print(f"m=0 에서   놓침 {missed_rate[0]:.3f} · 소란 {noisy_rate[0]:.3f}   (판당 칸 수)")
print(f"m=0.10 에서 놓침 {missed_rate[i10]:.3f} · 소란 {noisy_rate[i10]:.3f}")
print()
print("⚠️ 코드가 붙인 출처를 쓰면 두 값 다 정확히 0 이다 — 참값이 그 판정이니까.")

### 🖐 그래프에서 볼 것

**`m=0` 에서 두 선이 정확히 0 에서 출발한다.** 어긋남이 없으면 G2 는 완벽하다.
그리고 **코드가 출처를 붙이면 `m` 이 얼마든 항상 이 점에 머문다** — 코드는
어느 노드를 불렀는지 알기 때문에 틀릴 방법이 없다. **b1 과 b2 는 둘 다 여기 있다.**

**빨간 선(놓침)이 파란 선(소란)보다 가파르다.** 왜?
어긋난 finding 은 원래 자리를 **비우고** 남의 자리를 **채운다.** 그런데 남의 자리는
이미 차 있을 수도 있어서(그 노드도 뭔가 찾았으면) 소란이 안 생긴다. 반면
비워진 자리는 **거의 항상** 놓침이 된다. **한 번의 어긋남이 대칭이 아니다.**

그리고 놓침은 이 프로젝트에서 가장 비싼 실수다 —
`confidence ≥ 0.6 → 자동 게시` 규칙 아래서 **찾아낸 critical 이 조용히 통과한다.**

---
# §4.5 · 🎛 샌드박스 — 손잡이를 직접 돌린다

여기는 구현이 아니라 **감**을 만드는 자리다. 값을 바꿔가며 다시 실행할 것.

### 🖐 바꾸기 전에 예측할 것

`P_FIND` 를 0.7 → 1.0 으로 올리면 (살아있는 노드가 **전부** 뭔가 찾는다) **소란**은?

- (a) 늘어난다 — finding 이 많아지니 엉뚱한 자리도 많아진다
- (b) 그대로다
- (c) **정확히 0 이 된다**

**왜 그렇게 생각했나 한 줄:**

(힌트: 어긋난 finding 이 남의 자리로 갈 때, 그 자리가 **이미 차 있으면** 판정이 바뀌나?)

그리고 같은 실험에서 **놓침**은 어느 쪽으로 가나? 두 값을 같이 보라 —
한쪽이 좋아지는 게 다른 쪽이 좋아진다는 뜻이 아니다.

In [ ]:
# ═══ 손잡이 — 여기만 바꿔가며 다시 실행할 것 ═══
M        = 0.15    # 어긋남률.        후보: 0.0 / 0.05 / 0.15 / 0.4     원래: 0.15
N_DEAD   = 1       # 죽는 노드 수.    후보: 0 / 1 / 2 / 3               원래: 1
P_FIND   = 0.7     # 찾을 확률.       후보: 0.25 / 0.7 / 1.0            원래: 0.7
TRIALS_S = 4000    # 반복 횟수.       후보: 200 / 4000 / 50000          원래: 4000
#
#  M        키우면 → 두 선이 얼마나 빨리 벌어지나
#  N_DEAD   키우면 → 노드가 적게 살아남으면 어긋남이 더 아픈가 덜 아픈가
#  P_FIND   1.0 으로 → 위 예측의 답. 자리가 다 차면 무슨 일이 생기나
#  TRIALS_S 200 으로 → 숫자가 얼마나 흔들리나 (비용 대 정밀도)

r = np.random.default_rng(7)
tot_m = tot_n = 0
for _ in range(TRIALS_S):
    dead, fs = one_trial(r, M, N_DEAD, P_FIND)
    a, b = score(dead, fs)
    tot_m += a
    tot_n += b

print(f"M={M} · N_DEAD={N_DEAD} · P_FIND={P_FIND} · {TRIALS_S}판")
print(f"  놓침 (찾았는데 '깨끗하다') : {tot_m/TRIALS_S:.3f} 칸/판")
print(f"  소란 (안 찾았는데 '찾았다') : {tot_n/TRIALS_S:.3f} 칸/판")
print()
print("  코드가 출처를 붙이면 두 줄 다 0.000 이다.")

---
# §5 · 그래서 D3 배선을 어떻게 하나

노트북이 실제로 좁혀준 것을 정리한다. **여기는 네가 채운다 — 아래 표를 완성해라.**

| 갈래 | G2 오판율 | 얻는 것 | 잃는 것 |
|---|---|---|---|
| **b0** 모델값을 그대로 믿음 | §4 의 빨간·파란 선 | | |
| **b1** 모델이 뱉고 코드가 덮어씀 | **0** | | |
| **b2** 스키마에서 빼고 코드가 붙임 | **0** | | |

**노트북이 답해준 것 셋:**

1. §1 — b1 의 "공짜 계측기" 논거는 **호출자가 갈린 뒤에야** 성립한다
2. §3 — 그 계측기가 말을 하려면 `n` 이 수백 필요하다. **공짜가 아니다**
3. §4 — **G2 관점에서 b1 과 b2 는 구별되지 않는다.** 갈리는 건 다른 축이다

**노트북이 답 못 하는 것** (여기부터는 판단이다):

- 모델에게 안 물으면 **추론 토큰이 줄어드나?** — 재본 적 없다
- `Finding(...)` 만으로 완전하지 않게 되는 것의 비용 — 코드를 읽는 사람의 부담
- 옛 `evals/runs/` 18판과 새 판의 모양이 갈리는 것 — `grader.py` 는 `agent_type` 을
  **안 본다**(확인함). 그래서 무해할 가능성이 높지만, 확인할 자리가 하나 더 있나?

> ### ✍️ 내 결정과 한 줄 근거
>
> (여기에 직접 쓴다. `docs/CURRENT.md` 의 「확정된 결정」 표에 그대로 옮길 문장으로.)

---
# §6 · 자가 점검

- [ ] `count_mismatch` 의 네 줄이 왜 전부 무의미한지 **한 문장으로** 말할 수 있다
- [ ] G2 가 "조용히 틀리는" 경로를 **에러 없이** 설명할 수 있다
- [ ] 한 번의 어긋남이 왜 **놓침과 소란을 비대칭으로** 만드는지 말할 수 있다
- [ ] `n=17` 에서 어긋남 0건일 때 **뭐라고 말하면 안 되는지** 안다
- [ ] b1 과 b2 를 가르는 축이 **G2 가 아니라는 것**을 안다. 그럼 무슨 축인가?
- [ ] **Claude 없이** 이 시뮬레이션을 다시 만들 수 있다 —
      참값을 정하고, 관측을 만들고, 채점한다. 세 단계다